En aquesta pràctica treballareu amb un model de detecció d’objectes basat en PyTorch i la xarxa neuronal VGG16. L'objectiu es modificar el model per adaptar-lo a un problema de detecció d'objectes específic utilitzant un conjunt de dades simples.

![Exemple](08_Detecció/imgs/img.png)

Emprarem un *dataset* de detecció d'objectes que conté imatges d'estrelles. Aquest conjunt de dades és senzill i ideal per a practicar tècniques de detecció d'objectes. El podeu trobar a Kaggle al següent [enllaç](https://www.kaggle.com/datasets/kishanj/simple-object-detection). Per carregar aquest tipus de dataset haurem d'implementar una classe personalitzada que hereti de `torch.utils.data.Dataset`.

In [10]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import glob
import pandas as pd
import numpy as np
from torchvision import transforms as T
import xml.etree.ElementTree as ET

class EstrellesDataset(Dataset):
    def __init__(self, transforms):
        super().__init__()
        self.transform = transforms
        self.root = "datasets" # ruta del directorio base
        self.image_dir = os.path.join(self.root, "images") # ruta del directorio de imagenes
        self.annotations_dir = os.path.join(self.root, "annotations") # ruta del directorio de anotaciones

        self.images = sorted(glob.glob(os.path.join(self.image_dir, "*.jpg"))) # lista todas las imagenes


    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # cargar imagen
        img_path = self.images[idx]
        img = Image.open(img_path).convert("RGB")

        # obtener nombre base para buscar el XML correspondiente
        xml_path = os.path.join(self.annotations_dir, os.path.basename(img_path).replace(".jpg", ".xml"))

        # cargar anotaciones
        boxes, labels = self.parse_xml(xml_path)

        # aplicar transformaciones a la imagen si existe
        if(self.transform):
            img = self.transform(img)

        # convertir boxes y labels a tensores
        bbox = torch.FloatTensor(boxes)
        label = torch.LongTensor(labels)

        return img, label, bbox
    
    # Funcion para parsear el XML y extraer las boxes
    def parse_xml(self, xml_file):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        boxes = []
        labels = []
        
        for obj in root.findall(".//object"):
            bndbox = obj.find("bndbox")
            xmin = float(bndbox.find("xmin").text)
            ymin = float(bndbox.find("ymin").text)
            xmax = float(bndbox.find("xmax").text)
            ymax = float(bndbox.find("ymax").text)
            
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(1)  # 1 para estrella
            
        return np.array(boxes), np.array(labels)

## Preparació del model

Començarem carregant el model VGG16 preentrenat i adaptant-lo per a la detecció d'objectes. Afegirem capes addicionals per predir les caixes delimitadores (bounding boxes) i les classes dels objectes.

In [11]:
from torch import nn
from torchvision import models, transforms

vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
backbone = vgg16.features

class VGG16ObjectDetector(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.avgpool = nn.AdaptiveAvgPool2d((7,7))

        # Congelar los pesos del backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Capes fully-connected de la VGG original
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(p=0.5)
        )

        # Cap de classificació
        self.class_head = nn.Sequential(nn.Linear(4096, 2)) # 2 classes, estrella o fons

        # Cap de regressió de bounding box
        self.bbox_head = nn.Sequential(nn.Linear(4096, 4)) # 4 valors: x1, y1, x2, y2

    def forward(self, x):
        # Extraer features con backbone
        x = self.backbone(x)
        
        # Aplicar average pooling
        x = self.avgpool(x)
        
        # Flatten
        x = self.flatten(x)
        
        # Pasar por fully-connected layers
        x = self.fc(x)
        
        # Obtener predicciones
        class_logits = self.class_head(x)
        bbox_preds = self.bbox_head(x)

        return class_logits, bbox_preds


Generar el model

In [12]:
import torch
# Crear el modelo
model = VGG16ObjectDetector(backbone=backbone)

batch_size = 4
dummy_input = torch.randn(batch_size, 3, 224, 224)  # Batch de 4 imágenes RGB 224x224
class_out, bbox_out = model(dummy_input)

print(f"Output shapes:")
print(f"- Class logits: {class_out.shape}")  # Should be [4, 2]
print(f"- Bbox predictions: {bbox_out.shape}") 

Output shapes:
- Class logits: torch.Size([4, 2])
- Bbox predictions: torch.Size([4, 4])
